In [ ]:
import data_loader
import numpy as np
from skimage.feature import hog
from skimage import color
from sklearn.preprocessing import StandardScaler
from sklearn import svm
from sklearn.metrics import f1_score, roc_curve, auc, precision_recall_curve, confusion_matrix, classification_report
from joblib import Parallel, delayed
import time
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.utils import resample
from collections import defaultdict


loading annotations into memory...
Done (t=0.14s)
creating index...
index created!
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!


In [2]:
train, val, test = data_loader.load_masks()


In [3]:
def sliding_window(image, step_size, window_size):
    for y in range(0, image.shape[0], step_size):
        for x in range(0, image.shape[1], step_size):
            yield (x, y, image[y:y+window_size[1], x:x+window_size[0]])


In [ ]:
def extract_hog_features(window):
    """Extract HOG features from a window patch."""
    if window.size == 0:
        return np.zeros(324)  # Default HOG feature size for 64x64 window
    
    # Skip windows that are too small for HOG
    if window.shape[0] < 16 or window.shape[1] < 16:
        return np.zeros(324)
    
    # Convert to grayscale
    if len(window.shape) == 3:
        gray = color.rgb2gray(window)
    else:
        gray = window
    
    # Extract HOG features
    try:
        features = hog(gray, orientations=9, pixels_per_cell=(8, 8), 
                       cells_per_block=(2, 2), feature_vector=True)
    except ValueError:
        # If HOG fails, return zero vector
        return np.zeros(324)
    
    return features

def process_window_hog(args):
    """Process a single window for parallel execution."""
    x, y, window_patch, mask_patch = args
    feature_vector = extract_hog_features(window_patch)
    # Multi-class: use majority category in patch (0=background, 1-6=damage types)
    damage_pixels = mask_patch[mask_patch > 0]
    if len(damage_pixels) > (mask_patch.size * 0.1):  # 10% threshold
        label = int(np.bincount(damage_pixels).argmax())  # Most common damage category
    else:
        label = 0  # Background
    return feature_vector, label


In [ ]:
# Parallel HOG feature extraction
n_jobs = 8
print(f"Using {n_jobs} CPU cores for parallel processing...")
start_time = time.time()

# Collect all windows and mask patches first
all_windows_data = []
print("Collecting windows from training images...")
for data in tqdm(train, desc="Collecting windows"):
    image = data['image']
    mask = data['mask']
    
    windows = list(sliding_window(image, 32, (64, 64)))
    for x, y, window_patch in windows:
        h, w = window_patch.shape[:2]
        mask_patch = mask[y:y+h, x:x+w]
        all_windows_data.append((x, y, window_patch, mask_patch))

print(f"Total windows collected: {len(all_windows_data)}")

# Extract labels first to balance classes (multi-class)
print("Pre-labeling windows for balanced sampling...")
pre_labels = []
for data in tqdm(all_windows_data, desc="Pre-labeling"):
    x, y, window_patch, mask_patch = data
    # Multi-class label extraction
    damage_pixels = mask_patch[mask_patch > 0]
    if len(damage_pixels) > (mask_patch.size * 0.1):  # 10% threshold
        label = int(np.bincount(damage_pixels).argmax())  # Most common damage category
    else:
        label = 0  # Background
    pre_labels.append(label)

# Separate by class (0=background, 1-6=damage types)
class_windows = defaultdict(list)
for w, l in zip(all_windows_data, pre_labels):
    class_windows[l].append(w)

print("Class distribution:")
for class_id in sorted(class_windows.keys()):
    print(f"  Class {class_id}: {len(class_windows[class_id])} windows")

# Sample 10% from each class, then balance
min_class_size = min(len(windows) for windows in class_windows.values())
sample_size = int(min_class_size * 0.1)  # 10% of the smallest class

print(f"Sampling {sample_size} windows from each class (balanced)...")

# Sample from each class
sampled_windows = []
for class_id in sorted(class_windows.keys()):
    class_data = class_windows[class_id]
    if len(class_data) >= sample_size:
        sampled = resample(class_data, n_samples=sample_size, random_state=42, replace=False)
        sampled_windows.extend(sampled)
        print(f"  Class {class_id}: sampled {len(sampled)} windows")

# Combine and shuffle
all_windows_data = sampled_windows
np.random.seed(42)
np.random.shuffle(all_windows_data)

print(f"Final balanced dataset: {len(all_windows_data)} windows ({len(class_windows)} classes)")

# Process in parallel with joblib (better progress bar support)
print(f"Processing {len(all_windows_data)} windows with {n_jobs} workers...")

# Use joblib with verbose output - it shows progress automatically
results = Parallel(n_jobs=n_jobs, verbose=10)(
    delayed(process_window_hog)(data) for data in tqdm(all_windows_data, desc="Extracting HOG features")
)

# Unpack results
all_features, all_labels = zip(*results)
X = np.array(all_features)
y = np.array(all_labels)

elapsed = time.time() - start_time
print(f"Extracted {len(X)} feature vectors in {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)")
print(f"Feature vector dimension: {X.shape[1]}")
print(f"Class distribution: {np.bincount(y)}")


Using 8 CPU cores for parallel processing...


Total windows collected: 1952768
Pre-labeling windows for balanced sampling...


Pre-labeling: 100%|██████████| 1952768/1952768 [00:27<00:00, 71531.18it/s] 


Background windows: 1343458
Damage windows: 609310
Sampling 60931 windows from each class (balanced)...
Final balanced dataset: 121862 windows (60931 background + 60931 damage)
Processing 121862 windows with 8 workers...


Extracting HOG features:   0%|          | 16/121862 [00:02<5:58:16,  5.67it/s][Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:    2.4s
[Parallel(n_jobs=8)]: Done   9 tasks      | elapsed:    2.4s
[Parallel(n_jobs=8)]: Done  16 tasks      | elapsed:    2.5s
Extracting HOG features:   0%|          | 32/121862 [00:02<2:23:20, 14.17it/s][Parallel(n_jobs=8)]: Done  25 tasks      | elapsed:    2.5s


ValueError: The input image is too small given the values of pixels_per_cell and cells_per_block. It should have at least: 16 rows and 16 cols.

Extracting HOG features:   0%|          | 32/121862 [00:17<2:23:20, 14.17it/s]

In [ ]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
# Train multi-class SVM classifier
clf = svm.SVC(decision_function_shape='ovr')  # One-vs-rest for multi-class
clf.fit(X_scaled, y)
print("Multi-class model trained!")


In [ ]:
def predict_image(image, model, scaler, step_size=32, window_size=(64, 64), n_jobs=8):
    """Predict on a single image using sliding window with HOG features (parallelized)."""
    windows = list(sliding_window(image, step_size, window_size))
    
    # Extract HOG features in parallel
    window_patches = [w[2] for w in windows]
    features = Parallel(n_jobs=n_jobs)(delayed(extract_hog_features)(patch) for patch in window_patches)
    
    features = np.array(features)
    
    # Scale and predict
    features_scaled = scaler.transform(features)
    predictions = model.predict(features_scaled)
    
    # Reconstruct mask (multi-class: use predicted category)
    mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)
    for (x, y, _), pred in zip(windows, predictions):
        if pred > 0:  # damage (any category > 0)
            h, w = window_size[1], window_size[0]
            mask[y:y+h, x:x+w] = pred  # Store category ID
    
    return mask


In [ ]:
# Extract test features and evaluate
print("Extracting test features...")
start_time = time.time()

test_windows_data = []
print("Collecting windows from test images...")
for data in tqdm(test, desc="Collecting test windows"):
    image = data['image']
    mask = data['mask']
    
    windows = list(sliding_window(image, 32, (64, 64)))
    for x, y, window_patch in windows:
        h, w = window_patch.shape[:2]
        mask_patch = mask[y:y+h, x:x+w]
        test_windows_data.append((x, y, window_patch, mask_patch))

print(f"Total test windows collected: {len(test_windows_data)}")

# Extract labels first to balance test classes (multi-class)
print("Pre-labeling test windows for balanced sampling...")
test_pre_labels = []
for data in tqdm(test_windows_data, desc="Pre-labeling test"):
    x, y, window_patch, mask_patch = data
    # Multi-class label extraction
    damage_pixels = mask_patch[mask_patch > 0]
    if len(damage_pixels) > (mask_patch.size * 0.1):  # 10% threshold
        label = int(np.bincount(damage_pixels).argmax())  # Most common damage category
    else:
        label = 0  # Background
    test_pre_labels.append(label)

# Separate by class (multi-class)
test_class_windows = defaultdict(list)
for w, l in zip(test_windows_data, test_pre_labels):
    test_class_windows[l].append(w)

print("Test class distribution:")
for class_id in sorted(test_class_windows.keys()):
    print(f"  Class {class_id}: {len(test_class_windows[class_id])} windows")

# Sample 10% from each class, then balance
test_min_class_size = min(len(windows) for windows in test_class_windows.values())
test_sample_size = int(test_min_class_size * 0.1)  # 10% of the smallest class

print(f"Sampling {test_sample_size} windows from each test class (balanced)...")

# Sample from each class
test_sampled_windows = []
for class_id in sorted(test_class_windows.keys()):
    class_data = test_class_windows[class_id]
    if len(class_data) >= test_sample_size:
        sampled = resample(class_data, n_samples=test_sample_size, random_state=42, replace=False)
        test_sampled_windows.extend(sampled)
        print(f"  Class {class_id}: sampled {len(sampled)} windows")

# Combine and shuffle
test_windows_data = test_sampled_windows
np.random.seed(42)
np.random.shuffle(test_windows_data)

print(f"Final balanced test dataset: {len(test_windows_data)} windows ({len(test_class_windows)} classes)")

# Process in parallel with joblib
print(f"Processing {len(test_windows_data)} test windows...")
test_results = Parallel(n_jobs=n_jobs, verbose=10)(
    delayed(process_window_hog)(data) for data in tqdm(test_windows_data, desc="Extracting test HOG features")
)

test_features, test_labels = zip(*test_results)
X_test = np.array(test_features)
y_test = np.array(test_labels)

X_test_scaled = scaler.transform(X_test)

elapsed = time.time() - start_time
print(f"Test features extracted in {elapsed:.1f} seconds")

# Get predictions
predictions = clf.predict(X_test_scaled)
decision_scores = clf.decision_function(X_test_scaled)


In [ ]:
# Comprehensive evaluation metrics
print("=" * 60)
print("EVALUATION METRICS")
print("=" * 60)

# F1 Score (multi-class: weighted average)
f1_macro = f1_score(y_test, predictions, average='macro')
f1_weighted = f1_score(y_test, predictions, average='weighted')
print(f"\nF1 Score (macro): {f1_macro:.4f}")
print(f"F1 Score (weighted): {f1_weighted:.4f}")

# Get category names
category_map = data_loader.get_categories(data_loader.train_coco)
class_names = [category_map.get(i, f'Class_{i}') for i in sorted(np.unique(np.concatenate([y_test, predictions])))]

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names=class_names, zero_division=0))

# Confusion Matrix
cm = confusion_matrix(y_test, predictions, labels=sorted(np.unique(np.concatenate([y_test, predictions]))))
print("\nConfusion Matrix:")
print(cm)

# Plot confusion matrix
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names,
            yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

# Multi-class ROC (one-vs-rest)
from sklearn.preprocessing import label_binarize
from itertools import cycle

# Binarize labels for multi-class ROC
classes = sorted(np.unique(np.concatenate([y_test, predictions])))
y_test_bin = label_binarize(y_test, classes=classes)
n_classes = len(classes)

# Get decision scores for all classes (one-vs-rest)
if hasattr(clf, 'decision_function'):
    decision_scores_bin = clf.decision_function(X_test_scaled)
    if decision_scores_bin.ndim == 1:  # Binary case
        decision_scores_bin = np.column_stack([-decision_scores_bin, decision_scores_bin])
else:
    # Use predict_proba if available
    decision_scores_bin = clf.predict_proba(X_test_scaled)

# Compute ROC for each class
fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], decision_scores_bin[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Compute micro-average ROC
fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), decision_scores_bin.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

plt.subplot(1, 2, 2)
colors = cycle(['aqua', 'darkorange', 'cornflowerblue', 'red', 'green', 'purple', 'brown'])
for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f'{class_names[i]} (AUC = {roc_auc[i]:.2f})')
plt.plot(fpr["micro"], tpr["micro"], color='deeppink', linestyle='--', lw=2,
         label=f'Micro-avg (AUC = {roc_auc["micro"]:.2f})')
plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC Curve')
plt.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

print(f"\nMicro-averaged ROC AUC: {roc_auc['micro']:.4f}")
for i, name in enumerate(class_names):
    print(f"  {name}: {roc_auc[i]:.4f}")

# Precision-Recall Curve (micro-averaged)
precision, recall, _ = precision_recall_curve(y_test_bin.ravel(), decision_scores_bin.ravel())
pr_auc = auc(recall, precision)

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, color='blue', lw=2, label=f'PR curve (AUC = {pr_auc:.4f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="lower left")
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"PR AUC: {pr_auc:.4f}")
print("=" * 60)
